In [26]:
import pandas as pd
import numpy as np

In [27]:
data = "../data/auto.json"
surnames_data = "../../datasets/surname.json"

## **auto.json**

In [28]:
df = pd.read_json(data, orient="records")

In [29]:
pd.options.display.float_format = "{:.2f}".format

## **Обогащение DataFrame сэмплом из этого же DataFrame**

### **Сэмпл из 200 новых наблюдений с random_state = 21**

In [30]:
sample = df.sample(n=200, random_state=21).copy()

sample["Refund"] = np.random.choice(df["Refund"].dropna().values, size=200)
sample["Fines"] = np.random.choice(df["Fines"].dropna().values, size=200)

### **DataFrame concat_rows**

In [31]:
concat_rows = pd.concat([df, sample], ignore_index=True)

In [32]:
concat_rows.head(20)

,CarNumber,Refund,Fines,Make,Model
0,Y163O8161RUS,2,3200.00,Ford,Focus
1,E432XX77RUS,1,6500.00,Toyota,Camry
2,7184TT36RUS,1,2100.00,Ford,Focus
3,X582HE161RUS,2,2000.00,Ford,Focus
4,92918M178RUS,1,5700.00,Ford,Focus
5,H234YH197RUS,2,6000.00,Ford,Focus
6,E40577152RUS,1,8594.59,Ford,Focus
7,707987163RUS,2,2200.00,Ford,Focus
8,K330T8197RUS,2,8200.00,Skoda,Octavia
9,X786CO96RUS,1,8594.59,Ford,Focus


In [51]:
concat_rows.count()

CarNumber    925
Refund       925
Fines        925
Make         925
Model        914
dtype: int64

## **Обогащение concat_rows новой колонкой с сгенерированными данными**

### **Вызов np.random.seed(21)**

In [33]:
np.random.seed(21)

### **Series с именем Year со случайными целыми числами от 1980 до 2019**

In [34]:
years = pd.Series(np.random.randint(1980, 2020, size=len(concat_rows)), name="Year")

### **DataFrame fines**

In [35]:
fines = pd.concat([concat_rows, years], axis=1)

In [36]:
fines.head(20)

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2,3200.00,Ford,Focus,1989
1,E432XX77RUS,1,6500.00,Toyota,Camry,1995
2,7184TT36RUS,1,2100.00,Ford,Focus,1984
3,X582HE161RUS,2,2000.00,Ford,Focus,2015
4,92918M178RUS,1,5700.00,Ford,Focus,2014
5,H234YH197RUS,2,6000.00,Ford,Focus,1990
6,E40577152RUS,1,8594.59,Ford,Focus,1988
7,707987163RUS,2,2200.00,Ford,Focus,2016
8,K330T8197RUS,2,8200.00,Skoda,Octavia,2018
9,X786CO96RUS,1,8594.59,Ford,Focus,2000


In [52]:
fines.count()

CarNumber    930
Refund       930
Fines        930
Make         930
Model        919
Year         930
dtype: int64

## **Обогащение DataFrame данными из другого DataFrame**

### **DataFrame с номерами автомобилей и их владельцами**

In [37]:
surnames_df = pd.read_json(surnames_data)
surnames = surnames_df.iloc[1:, 0].tolist()

In [38]:
unique_cars = fines["CarNumber"].nunique()
selected_surnames = np.random.choice(surnames, size=unique_cars)

In [39]:
owners = pd.DataFrame(
    {"CarNumber": fines["CarNumber"].unique(), "SURNAME": selected_surnames}
)

In [40]:
owners.head(20)

,CarNumber,SURNAME
0,Y163O8161RUS,BAKER
1,E432XX77RUS,CRUZ
2,7184TT36RUS,MARTIN
3,X582HE161RUS,REED
4,92918M178RUS,COOPER
5,H234YH197RUS,WOOD
6,E40577152RUS,COLLINS
7,707987163RUS,ALLEN
8,K330T8197RUS,PRICE
9,X786CO96RUS,ROBINSON


### **Добавление ещё 5-ти наблюдений в fines**

In [41]:
new_rows = pd.DataFrame(
    {
        "CarNumber": [
            "X123XX197RUS",
            "Y456YY197RUS",
            "Z789ZZ197RUS",
            "W111WW197RUS",
            "V222VV197RUS",
        ],
        "Refund": [1, 2, 1, 2, 1],
        "Fines": [5000, 12000, 3000, 8000, 15000],
        "Make": ["Ford", "Toyota", "Honda", "Ford", "Toyota"],
        "Model": ["Focus", "Camry", "Civic", "Focus", "Corolla"],
        "Year": [2020, 2021, 2020, 2022, 2021],
    }
)

fines = pd.concat([fines, new_rows], ignore_index=True)

### **Удаление последних 20-ти наблюдений из owners и добавление трех новых, отличных от добавленных в fines**

In [42]:
owners = owners.iloc[:-20]

new_owners = pd.DataFrame(
    {
        "CarNumber": ["QQ111QQ197RUS", "WW222WW197RUS", "EE333EE197RUS"],
        "SURNAME": ["SMITH", "JOHNSON", "WILLIAMS"],
    }
)

owners = pd.concat([owners, new_owners], ignore_index=True)

### **Соединение двух DataFrame**

#### **Только** те номера, которые есть **в обоих** DataFrame

In [53]:
inner_join = pd.merge(fines, owners, on="CarNumber", how="inner")

#### **Все** номера **из обоих** DataFrame.

In [44]:
outer_join = pd.merge(fines, owners, on="CarNumber", how="outer")

#### **Только** номера из fines

In [45]:
left_join = pd.merge(fines, owners, on="CarNumber", how="left")

#### **Только** номера из owners

In [46]:
right_join = pd.merge(fines, owners, on="CarNumber", how="right")

## **Сводная таблица (pivot table) из fines**

In [47]:
pd.pivot_table(
    data=fines, values="Fines", index=["Make", "Model"], columns="Year", aggfunc="sum"
)

Year                   1980      1981      1982     1983      1984      1985  \
Make       Model                                                               
Ford       Focus   89894.59 394078.35 140678.35 72400.00 114094.59 160383.76   
           Mondeo       NaN       NaN       NaN      NaN       NaN       NaN   
Honda      Civic        NaN       NaN       NaN      NaN       NaN       NaN   
Skoda      Octavia 43600.00       NaN   7400.00 11594.59       NaN  10294.59   
Toyota     Camry   12000.00   8594.59       NaN  7200.00       NaN       NaN   
           Corolla      NaN       NaN   2000.00      NaN       NaN       NaN   
Volkswagen Golf    30900.00       NaN       NaN  8594.59    300.00  24000.00   
           Jetta        NaN       NaN       NaN      NaN       NaN       NaN   
           Passat       NaN   1600.00       NaN  3200.00  10000.00   5000.00   
           Touareg      NaN       NaN       NaN      NaN       NaN   5800.00   

Year                    1986      1987     1988      1989  ...      2013  \
Make       Model                                           ...             
Ford       Focus   108194.59 106100.00 92394.59 107794.59  ... 168594.59   
           Mondeo        NaN       NaN      NaN   8600.00  ...       NaN   
Honda      Civic         NaN       NaN      NaN       NaN  ...       NaN   
Skoda      Octavia    600.00   5200.00  4500.00  91400.00  ...  12594.59   
Toyota     Camry         NaN       NaN      NaN  22400.00  ...       NaN   
           Corolla       NaN   9000.00      NaN   4000.00  ...       NaN   
Volkswagen Golf          NaN  17400.00      NaN   5800.00  ...   5000.00   
           Jetta         NaN       NaN      NaN       NaN  ...       NaN   
           Passat   15000.00  12300.00      NaN       NaN  ...       NaN   
           Touareg       NaN       NaN      NaN       NaN  ...       NaN   

Year                    2014      2015     2016      2017      2018     2019  \
Make       Model                                                               
Ford       Focus   166994.59 260194.59 82889.17 244900.00 279294.59 61200.00   
           Mondeo        NaN       NaN 46200.00       NaN       NaN      NaN   
Honda      Civic         NaN       NaN      NaN       NaN       NaN      NaN   
Skoda      Octavia    300.00  46394.59   300.00  13900.00 156200.00  9500.00   
Toyota     Camry     8594.59       NaN      NaN       NaN  40000.00 18100.00   
           Corolla       NaN       NaN  1200.00   9600.00 168000.00      NaN   
Volkswagen Golf          NaN   2300.00      NaN       NaN    600.00      NaN   
           Jetta         NaN       NaN      NaN       NaN       NaN      NaN   
           Passat        NaN    600.00  2100.00       NaN       NaN      NaN   
           Touareg   1300.00    500.00      NaN       NaN       NaN      NaN   

Year                  2020     2021    2022  
Make       Model                             
Ford       Focus   5000.00      NaN 8000.00  
           Mondeo      NaN      NaN     NaN  
Honda      Civic   3000.00      NaN     NaN  
Skoda      Octavia     NaN      NaN     NaN  
Toyota     Camry       NaN 12000.00     NaN  
           Corolla     NaN 15000.00     NaN  
Volkswagen Golf        NaN      NaN     NaN  
           Jetta       NaN      NaN     NaN  
           Passat      NaN      NaN     NaN  
           Touareg     NaN      NaN     NaN  

[10 rows x 43 columns]

## **Сохранение fines и owners в CSV-файлы без индекса**


In [48]:
fines.to_csv("fines.csv", index=False)
owners.to_csv("owners.csv", index=False)